# CML3113 Engineering ML project: REHEARSAL (unmarked), predicting lagging reactive power (kVarh)
Run **Setup: run me first**, then do each task in order and store the results it names. Practise with a 3-hour timer.
There is no Submit button: your notebook saves itself; on the entry page use Check my work and I have finished. Nothing is locked or marked.
No AI tools and no help from other people, as in the project.
Add as many cells as you like under a task; plain code is fine (no functions needed).

In [ ]:
#@title Setup: run me first
# IUEA-SETUP-CELL (supplied). Run it first, and again after a restart. The marking computer always uses the original of this cell.
_IUEA_IDENTITY = {"course": "CML3113", "test": "ML_REHEARSAL"}
import base64, bz2, hashlib, io, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, precision_score, recall_score
FULL_NAME = _IUEA_IDENTITY.get("name", "")
STUDENT_ID = _IUEA_IDENTITY.get("registration", "")
COHORT = _IUEA_IDENTITY.get("course", "")
TEST_ID = _IUEA_IDENTITY.get("test", "")
_IUEA_ALLOCATION = _IUEA_IDENTITY.get("allocation", "")
_IUEA_DEADLINE = _IUEA_IDENTITY.get("deadline", "")
_PACKED = '''
...
wLuVJT+P/xdyRThQkAH6Stg=
'''
_bytes = bz2.decompress(base64.b64decode("".join(_PACKED.split())))
if hashlib.sha256(_bytes).hexdigest() != 'f44f5c5672c2347798f8155e4aeebd5e1a720cf85dd125bb776d37ba77d34fa5':
    raise RuntimeError("The data in this cell is damaged. Tell your lecturer.")
_data = json.loads(_bytes)
TARGET = 'Lagging_Current_Reactive.Power_kVarh'
TARGET_UNIT = 'kVarh'
FEATURES = ['Usage_kWh', 'Leading_Current_Reactive_Power_kVarh', 'NSM', 'weekend']
LIMIT = 40.0
THRESHOLD = 0.5
MAX_MAE, MAX_MS = 6.0, 50
LATENCY_MS = {'baseline': 1, 'linear': 3, 'tree': 30}
raw = pd.read_csv(io.StringIO(_data["development"]))
_TEST_ROWS = pd.read_csv(io.StringIO(_data["test"]))
def choose_model(maes):
    ok = [m for m in maes if maes[m] is not None and maes[m] <= MAX_MAE and LATENCY_MS[m] <= MAX_MS]
    return min(ok, key=lambda m: (maes[m], LATENCY_MS[m], m)) if ok else None
TEST_OPEN_LOG = []
def open_test_set(model_name):
    if model_name not in ("baseline", "linear", "tree"):
        raise ValueError('use "baseline", "linear" or "tree"')
    if TEST_OPEN_LOG or os.path.exists("test_opened.json"):
        print("WARNING: THE TEST SET WAS ALREADY OPENED. Opening it again means it is no longer an honest test.")
    json.dump({"model": model_name}, open("test_opened.json", "w"))
    TEST_OPEN_LOG.append([model_name, THRESHOLD])
    return _TEST_ROWS.copy()
print("raw:", raw.shape, "| TARGET:", TARGET, "| FEATURES:", FEATURES)

# Automatic backup (supplied). Inside Colab, a copy of the notebook on your screen goes to your lecturer every 5 minutes.
# You do not need to do anything. Your work is collected automatically at your deadline.
def _iuea_backup_start(address='https://iuea-test-entry.jothammartin-wambi.workers.dev/api/backup', minutes=5):
    import json, threading, time, urllib.request
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or "google.colab" not in type(shell).__module__:
            return                                   # not in Colab (the marking computer, Check my notebook): no backup
        from google.colab import _message, output
        from IPython.display import display, HTML
    except Exception:
        return
    attempt = str(globals().get("_IUEA_ALLOCATION", ""))
    if not attempt:
        print("Backup is off: run the first cell (your details and the countdown) first, then this cell again.")
        return
    old = globals().get("_iuea_backup_thread")
    if old is not None and old.is_alive():
        return
    note = display(HTML(""), display_id=True)
    packed = _PACKED.strip()
    def send():
        notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
        for cell in notebook.get("cells", []):
            text = "".join(cell.get("source", "")) if isinstance(cell.get("source"), list) else str(cell.get("source", ""))
            if "# IUEA-SETUP-CELL" in text:
                cell["source"] = text.replace(packed, "__IUEA_" + "PACKED_DATA__")
        href = str(output.eval_js("location.href"))[:500]
        body = json.dumps({"attempt": attempt, "href": href, "notebook": json.dumps(notebook)}).encode("utf-8")
        request = urllib.request.Request(address, data=body, headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(request, timeout=60) as reply:
            return json.loads(reply.read().decode("utf-8"))
    def loop():
        while True:
            try:
                answer = send() or {}
                result = answer.get("result") or {}
                if result.get("other_file"):
                    note.update(HTML('<div style="border:3px solid #b3261e;background:#fff0ee;padding:10px;font-size:16px;color:#7a1712">'
                                     '<b>You are working in a different file from your issued notebook.</b> Your work here is still backed up. '
                                     'Go back to the entry page and click Resume, or keep working here and tell your lecturer on the class channel.</div>'))
                elif answer.get("ok"):
                    note.update(HTML('<div style="color:#526577;font-size:13px">Automatic backup sent at ' + time.strftime("%H:%M") + '. Your notebook also saves itself; press Ctrl+S now and then.</div>'))
                elif result.get("closed") or "deadline" in str(answer.get("error", "")):
                    note.update(HTML('<div style="color:#526577;font-size:13px">Your time has ended. Your work was collected at your deadline.</div>'))
                    return
            except Exception:
                pass
            time.sleep(minutes * 60)
    thread = threading.Thread(target=loop, name="iuea-backup", daemon=True)
    thread.start()
    globals()["_iuea_backup_thread"] = thread
_iuea_backup_start()

def _iuea_countdown_start():
    import datetime
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or "google.colab" not in type(shell).__module__ or not _IUEA_DEADLINE:
            return
    except Exception:
        return
    end = datetime.datetime.fromisoformat(_IUEA_DEADLINE.replace("Z", "+00:00"))
    local = (end + datetime.timedelta(hours=3)).strftime("%H:%M")
    practice = TEST_ID == "ML_REHEARSAL"
    def line(*_):
        left = (end - datetime.datetime.now(datetime.timezone.utc)).total_seconds()
        if left <= 0:
            print("⏱ time is up (deadline " + local + " Kampala)" + ("" if practice else ": your work is collected"))
            return
        m = int(left // 60)
        text = "⏱ %d:%02d left (deadline %s Kampala)" % (m // 60, m % 60, local)
        if m < 30:
            text += "  Press Ctrl+S" + (" now: 10 minutes or less" if m < 10 else "")
        print(text)
    old = globals().get("_iuea_countdown")
    if old is not None:
        try:
            shell.events.unregister("post_run_cell", old)
        except Exception:
            pass
    shell.events.register("post_run_cell", line)
    globals()["_iuea_countdown"] = line
_iuea_countdown_start()


## Task 1: Clean the data (7 marks)
Make `clean` from `raw`, rows in the same order: add `timestamp` from `date` (`%d/%m/%Y %H:%M`, plus one day when the time is 00:00), `month` (the month as written in `date`) and `weekend` (1 when `WeekStatus` is "Weekend", else 0); drop the column `CO2(tCO2)`.
Drop a repeated `timestamp` (keep the first), a missing `TARGET`, the all-zero reading (`Usage_kWh` and both power factors 0), and impossible values (energy or reactive power below 0, a power factor outside 0 to 100, `NSM` outside 0 to 85500). Do not change `raw`.
Chart 1: a scatter of `Usage_kWh` (across) against `TARGET` (up) from `clean`, with a title, both axis names and the unit (kVarh).

**Input:** `raw`. **Store:** `clean`.

In [ ]:
# Your code here. Store: clean


### Note: split by month
- Readings 15 minutes apart are almost the same, so a random split puts near-copies of training rows in the validation set.
- The score then looks better than it will be next month: that is **leakage**.
- Split by month: train January to August, validate September and October, keep November and December for the test.
- Slides: Class 3, "Train-test contamination: validation rows in the training set make the score look perfect".

## Task 2: Split by month (2 marks)
Make `train` (months 1 to 8) and `valid` (months 9 and 10) from `clean`, rows in the same order. Use `month`, never a random split.

**Input:** `clean`. **Store:** `train`, `valid`.

In [ ]:
# Your code here. Store: train, valid


### Note: fit on the training rows only
- Every number learned from data (the imputation medians, the baseline mean) comes from `train` only, then is applied to `valid`.
- `make_pipeline(SimpleImputer(strategy="median"), model)` does this: it learns in `fit` and reuses in `predict`.
- Learning from all rows lets the model see the rows it is judged on, so the error looks too small.
- Slides: Class 3, "The pipeline rule: fit on the training set only, then apply the same numbers".

## Task 3: Three models, MAE and RMSE (13 marks)
Fit each model on `train` only (columns `FEATURES`, target `TARGET`) and predict `valid`: `make_pipeline(SimpleImputer(strategy="median"), model)` with `DummyRegressor()`, `LinearRegression()` and `DecisionTreeRegressor(max_depth=4, random_state=17)`.
Then the MAE and RMSE of each on `valid`.
Chart 2: a bar chart of the three MAE values, with a title, both axis names and the unit (kVarh).

**Input:** `train`, `valid`. **Store:** `baseline_pred`, `linear_pred`, `tree_pred`, `baseline_mae`, `linear_mae`, `tree_mae`, `baseline_rmse`, `linear_rmse`, `tree_rmse`.

In [ ]:
# Your code here. Store: baseline_pred, linear_pred, tree_pred, baseline_mae, linear_mae, tree_mae, baseline_rmse, linear_rmse, tree_rmse


### Note: logistic regression, confusion matrix, precision, recall
- **Logistic regression** gives each row a probability between 0 and 1 (`predict_proba(X)[:, 1]`); warn when it is at least 0.5.
- **Confusion matrix:** TP right warning, FP false alarm, FN missed warning, TN rightly none; `confusion_matrix(actual, warned, labels=[0, 1]).ravel()` gives tn, fp, fn, tp.
- **Precision** = TP / (TP + FP); **recall** = TP / (TP + FN); a zero denominator gives 0.
- Slides: Class 2, "Logistic regression in three lines: score, probability, prediction" and "The confusion matrix counts true and false positives and negatives".

## Task 4: Which readings should raise a warning? (3 marks)
A warning is a reading with `TARGET` at or above `LIMIT`. Fit `make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))` on `train` and store the probability of a warning for each `valid` row.
Warn when the probability is at least `THRESHOLD` (0.5), compare with the real warnings in `valid`, and store the counts, precision and recall (0 when a count is zero).

**Input:** `train`, `valid`. **Store:** `warning_proba`, `tp`, `fp`, `fn`, `tn`, `precision`, `recall`.

In [ ]:
# Your code here. Store: warning_proba, tp, fp, fn, tn, precision, recall


## Task 5: Choose one model
Use the supplied rule (MAE at most `MAX_MAE`, time at most `MAX_MS`, then the smallest MAE): `chosen_model = choose_model({"baseline": baseline_mae, "linear": linear_mae, "tree": tree_mae})`. Never use the test set to choose.

**Input:** `baseline_mae`, `linear_mae`, `tree_mae`. **Store:** `chosen_model`.

In [ ]:
# Your code here. Store: chosen_model


### Note: open the test set once, then refit
- The validation set chooses the model; the test set gives **one honest estimate** of the chosen one.
- Looking at the test set and then changing your choice makes its score too good to trust.
- Refit on `train` and `valid` together (more data), then measure on `test_rows` once.
- Slides: Class 3, "We choose max_depth on the validation set and keep the test set sealed".

## Task 6: Open the test set once (2 marks)
After Task 5, run `test_rows = open_test_set(chosen_model)` ONCE. Refit the chosen model and the warning model on `train` and `valid` together, then store the MAE on `test_rows` and the recall of the warnings at 0.5.

**Input:** `train`, `valid`, `chosen_model`. **Store:** `test_rows`, `test_mae`, `test_recall`.

In [ ]:
# Your code here. Store: test_rows, test_mae, test_recall


## Task 7: Part B, the decision memo form (15 automatic marks)
Fill in `MEMO` in the next cell, for the plant's energy manager. Use your own numbers, at least two decimals, in kVarh.

| Field | Enter | Marks |
|---|---|---|
| `recommended_model` | `"baseline"`, `"linear"` or `"tree"`: the model the rule chose and you opened the test set with. | 3 |
| `validation_mae` | The validation MAE of each model, from your notebook. | 3 |
| `test_mae`, `test_recall` | The MAE and the warning recall (0 to 1) from your one test opening. | 2 |
| `improvement_over_baseline` | Baseline validation MAE minus your model's (0 if you recommend the baseline). | 1 |
| `limits` | Three different letters: the true limits of this study. One mark each. | 3 |
| `next_step` | One letter: the best next step. | 1 |
| `answer_choice`, `answer_test_set` | Two answers in your own words, 8 to 80 words each. One mark each. | 2 |

**Limits** (some statements are false):
- **A.** The test set is only November and December of the same year, so the test metrics are one estimate that may be luckier or unluckier than the truth.
- **B.** The model with the lowest training error is always the best model for new months.
- **C.** The client limits (the error limit and the time limit) and the warning limit are invented teaching numbers; a real client would set their own.
- **D.** The baseline model should be left out of the comparison because it uses no features.
- **E.** Because the test set was opened only once, the test metrics have no uncertainty at all.
- **F.** All the readings come from one steel plant in one year (2018), so the model may not hold for another plant or for later years.
- **G.** Fitting the imputer on the training set only makes the validation results unreliable.
- **H.** The validation set is only two months (September and October), so the validation errors could change a lot with two other months.
- **I.** The test MAE proves that the model will be this accurate in every month of every year.
- **J.** RMSE is always smaller than MAE, so MAE is the more severe measure and should be the only one reported.

**Next step:**
- **A.** Collect readings from more years (or another plant) and check the chosen model again on months it has never seen before any decision.
- **B.** Move the test months into the training set so the model learns from more rows, and report the training error.
- **C.** Deploy the chosen model now, because the test MAE is below the client limit.
- **D.** Tune the maximum depth of the decision tree until its test MAE is lowest, then report that test MAE.
- **E.** Remove the baseline model from the report so the results look better.

**Written answers:**
- `answer_choice`: In one or two sentences: why do you recommend this model? Use a number from the validation set and say what you compared it with.
- `answer_test_set`: In one or two sentences: why is the test set opened only once, at the very end, and never used to choose the model?

In [ ]:
# Fill in every field; do not rename a field.
MEMO = {
    "recommended_model": "",                  # "baseline", "linear" or "tree"
    "validation_mae": {"baseline": None, "linear": None, "tree": None},   # your validation MAE values, in kVarh
    "test_mae": None,                         # the test MAE from your ONE opening, in kVarh
    "test_recall": None,                      # the test recall from your ONE opening (0 to 1)
    "improvement_over_baseline": None,        # baseline validation MAE minus your recommended model's, in kVarh
    "limits": ["", "", ""],                   # three different letters from the Limits list
    "next_step": "",                          # one letter from the Next step list
    "answer_choice": "",                      # one or two sentences
    "answer_test_set": "",                    # one or two sentences
}
